# Astrolabio: modelo diario de BTCUSDT

**Del pronóstico de rendimientos a las bases de un agente de trading.** Activo: BTCUSDT diario en MEXC.

**Cómo ejecutarlo.** En Colab elige *Entorno de ejecución > Cambiar tipo > GPU T4* y después *Ejecutar todo*. Si Colab no asigna una T4, el notebook se detiene antes de entrenar. Al terminar, descarga `resultados_t4.zip`, descomprímelo y copia su carpeta `t4-AAAA-MM-DD` a `models/` del proyecto.

**Orden.** Cinco etapas: datos y referencias, redes, validación temporal, diseño del agente, e indicadores con evaluación final. TE solo se abre una vez, en la etapa 5, con el sistema congelado.

## Entorno y evidencia de T4

In [ ]:
import json, os, platform, subprocess, sys, time, zipfile
from pathlib import Path

PRUEBA = os.environ.get('NB_PRUEBA') == '1'  # solo para probar el notebook sin GPU, con datos sintéticos; no es evidencia válida
SEMILLA_GLOBAL = 11
os.makedirs('bot', exist_ok=True); os.makedirs('research', exist_ok=True); os.makedirs('resultados', exist_ok=True)
for ruta in ('bot/__init__.py', 'research/__init__.py'):
    Path(ruta).touch()
SALIDA = Path('resultados')
inicio_notebook = time.time()
bitacora = []

def anotar(etapa, error, causa, accion, resultado):
    """Bitácora de errores: fecha, etapa, error, causa, acción correctiva y resultado de la verificación."""
    bitacora.append(dict(fecha=time.strftime('%Y-%m-%d %H:%M:%S'), etapa=etapa, error=error, causa=causa, accion_correctiva=accion, resultado_verificacion=resultado))

In [ ]:
evidencia = dict(python=platform.python_version(), semilla=SEMILLA_GLOBAL, inicio=time.strftime('%Y-%m-%d %H:%M:%S'), prueba_sin_gpu=PRUEBA)
if PRUEBA:
    print('MODO PRUEBA: sin T4 y con datos sintéticos. Esta corrida NO sirve como evidencia de T4.')
    evidencia |= dict(gpu='ninguna', valido_para_entrega=False)
else:
    try:
        salida = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=True).stdout
    except (FileNotFoundError, subprocess.CalledProcessError) as e:
        anotar('entorno', 'nvidia-smi no disponible', str(e), 'Detener el entrenamiento y cambiar el entorno a GPU T4', 'Sin entrenar')
        raise RuntimeError('No hay GPU en esta sesión. Elige Entorno de ejecución > Cambiar tipo > GPU T4 y vuelve a ejecutar.') from e
    print(salida)
    (SALIDA / 'nvidia_smi.txt').write_text(salida)
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    nombre = tf.config.experimental.get_device_details(gpus[0]).get('device_name', '') if gpus else ''
    print('GPU detectada:', nombre or 'ninguna')
    if 'T4' not in nombre:
        anotar('entorno', f'GPU distinta de T4: {nombre or "ninguna"}', 'Colab no garantiza el tipo de GPU', 'Detener el entrenamiento y reprogramar la ejecución', 'Sin entrenar')
        raise RuntimeError('No se asignó una T4: se detiene el entrenamiento profundo. Reprograma la ejecución.')
    with tf.device('/GPU:0'):
        a = tf.random.normal((2000, 2000))
        operacion = tf.linalg.matmul(a, a)
    print('Operación ejecutada en:', operacion.device)
    tf.config.experimental.reset_memory_stats('GPU:0')
    evidencia |= dict(gpu=nombre, tensorflow=tf.__version__, dispositivo_operacion=operacion.device, valido_para_entrega=True)
import keras
evidencia |= dict(keras=keras.__version__, backend=keras.backend.backend())
print(evidencia)

In [ ]:
# Guarda los resultados fuera del almacenamiento temporal de Colab
DRIVE = None
if not PRUEBA:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE = Path('/content/drive/MyDrive/DL_Finanzas_T4')
        DRIVE.mkdir(parents=True, exist_ok=True)
    except Exception as e:  # sin Drive se sigue, pero queda registrado
        anotar('entorno', 'Google Drive no disponible', str(e), 'Descargar el zip al terminar', 'Pendiente')
        print('Sin Drive:', e, '- descarga el zip al terminar.')

### Código del proyecto

Estas celdas escriben los mismos módulos que usa el agente. Así las variables se calculan igual al entrenar y al operar. Se pueden leer y explicar aquí mismo.

In [ ]:
%%writefile bot/data.py
"""Auditoría, limpieza y partición de velas diarias. Nada aquí mira el futuro ni inventa filas."""
import hashlib

import numpy as np
import pandas as pd

DIA_MS = 86_400_000
COLUMNAS = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume']
PRECIOS = ['open', 'high', 'low', 'close']


class DataError(ValueError):
    """Datos que impiden continuar. El mensaje explica qué revisar."""


def fecha(ms):
    return pd.Timestamp(int(ms), unit='ms', tz='UTC').strftime('%Y-%m-%d')


def frame_from_rows(filas):
    """Convierte las filas de /api/v3/klines en un DataFrame, sin corregir nada todavía."""
    if not filas:
        raise DataError('Conjunto vacío: la fuente no devolvió velas. Revisa símbolo, intervalo y conexión.')
    if len(filas[0]) < len(COLUMNAS):
        raise DataError(f'Fila con {len(filas[0])} columnas; se esperaban {len(COLUMNAS)}: {COLUMNAS}.')
    df = pd.DataFrame([f[:len(COLUMNAS)] for f in filas], columns=COLUMNAS)
    for col in COLUMNAS:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['date'] = pd.to_datetime(df.open_time, unit='ms', utc=True).dt.strftime('%Y-%m-%d')
    return df


def audit(df):
    """Describe los problemas de la serie sin modificarla."""
    invalidos = []
    for col in PRECIOS + ['volume']:
        malos = ~np.isfinite(df[col]) | (df[col] <= 0 if col in PRECIOS else df[col] < 0)
        invalidos += [{'columna': col, 'fecha': d, 'valor': v} for d, v in zip(df.date[malos], df[col][malos])]
    orden = df.drop_duplicates('open_time', keep='last').sort_values('open_time')
    saltos = orden.open_time.diff()
    huecos = [{'desde': a, 'hasta': b, 'dias': float(s) / DIA_MS}
              for a, b, s in zip(orden.date.shift(), orden.date, saltos) if pd.notna(s) and s >= 2 * DIA_MS]
    incoherentes = df[(df.high < df[['open', 'close']].max(axis=1)) | (df.low > df[['open', 'close']].min(axis=1))]
    return {
        'filas': len(df),
        'primera': orden.date.iloc[0],
        'ultima': orden.date.iloc[-1],
        'duplicadas': int(df.open_time.duplicated().sum()),
        'desordenadas': int((df.open_time.diff() < 0).sum()),
        'invalidos': invalidos,
        'huecos': huecos,
        'desalineadas': int((orden.open_time % DIA_MS != 0).sum()),
        'incoherentes': incoherentes.date.tolist(),
    }


def clean(df):
    """Regla documentada: quitar duplicados y filas inválidas, y conservar el tramo continuo más largo.

    No se rellena ningún día. Las velas con otro corte diario quedan fuera porque no miden lo mismo.
    Devuelve el DataFrame y la bitácora de lo que cambió.
    """
    bitacora = []
    inicial = len(df)
    salida = df.sort_values('open_time', kind='stable').drop_duplicates('open_time', keep='last')
    if len(salida) != inicial:
        bitacora.append(f'{inicial - len(salida)} duplicada(s) eliminada(s), se conserva la última versión.')
    validas = np.isfinite(salida[PRECIOS]).all(axis=1) & (salida[PRECIOS] > 0).all(axis=1) & (salida.volume >= 0)
    if (~validas).any():
        bitacora.append(f'{int((~validas).sum())} fila(s) con precio no válido retirada(s): {salida.date[~validas].tolist()}.')
    salida = salida[validas]
    alineadas = salida.open_time % DIA_MS == 0
    if (~alineadas).any():
        bitacora.append(f'{int((~alineadas).sum())} vela(s) con otro corte diario retirada(s).')
    salida = salida[alineadas]
    if salida.empty:
        raise DataError('Después de limpiar no queda ninguna vela válida. Revisa la fuente de datos.')
    tramo = (salida.open_time.diff() != DIA_MS).cumsum()
    mejor = tramo.value_counts().idxmax()
    continuo = salida[tramo == mejor].reset_index(drop=True)
    if len(continuo) != len(salida):
        bitacora.append(f'{len(salida) - len(continuo)} vela(s) fuera del tramo continuo más largo '
                        f'({continuo.date.iloc[0]} a {continuo.date.iloc[-1]}).')
    return continuo, bitacora


def add_target(df):
    """y_t = P_{t+1} / P_t - 1. La última fila queda sin objetivo observable."""
    salida = df.copy()
    salida['y'] = salida.close.shift(-1) / salida.close - 1
    return salida


def dataset_hash(df):
    contenido = df[COLUMNAS].to_csv(index=False).encode()
    return hashlib.sha256(contenido).hexdigest()


def split_indices(n, tr=0.70, va=0.15):
    """Partición cronológica por orden de la fecha objetivo. No hay división aleatoria."""
    n_tr, n_va = int(n * tr), int(n * va)
    if n_tr < 1 or n_va < 1 or n - n_tr - n_va < 1:
        raise DataError(f'Ejemplos insuficientes para TR/VA/TE: {n}.')
    return {'TR': np.arange(0, n_tr), 'VA': np.arange(n_tr, n_tr + n_va), 'TE': np.arange(n_tr + n_va, n)}

In [ ]:
%%writefile bot/features.py
"""Variables del modelo. La misma definición se usa al entrenar y en el agente.

Cada indicador usa solo historia disponible hasta t. Los primeros valores quedan vacíos.
"""
from collections import namedtuple

import numpy as np
import pandas as pd

from bot.data import DataError

Ventanas = namedtuple('Ventanas', 'X y inicio emision objetivo')

# Qué mide cada indicador y su fórmula; el notebook y el panel muestran esta tabla.
INDICADORES = {
    'dist_sma20': 'Distancia al promedio de 20 cierres: cierre / SMA20 - 1',
    'vol20': 'Desviación muestral de 20 rendimientos diarios, sin anualizar',
    'rsi14': 'RSI de 14 con suavizado de Wilder; 100 sin pérdidas, 50 sin movimiento',
    'macd_hist': 'MACD (EMA12 - EMA26) menos su señal EMA9, dividido entre el cierre',
    'bb_pos': 'Posición en bandas de Bollinger: (cierre - SMA20) / (2 desviaciones de 20 cierres)',
    'atr14': 'ATR de 14 con suavizado de Wilder, dividido entre el cierre',
    'vol_rel': 'Logaritmo del volumen entre su promedio de 20 días',
}


def _wilder(serie, periodo):
    return serie.ewm(alpha=1 / periodo, adjust=False, min_periods=periodo).mean()


def add_indicators(df):
    s = df.copy()
    cierre = s.close
    s['ret'] = cierre.pct_change(fill_method=None)
    s['sma20'] = cierre.rolling(20).mean()
    desv20 = cierre.rolling(20).std(ddof=1)
    s['dist_sma20'] = cierre / s.sma20 - 1
    s['vol20'] = s.ret.rolling(20).std(ddof=1)

    delta = cierre.diff()
    ganancia = _wilder(delta.clip(lower=0), 14)
    perdida = _wilder(-delta.clip(upper=0), 14)
    rsi = 100 - 100 / (1 + ganancia / perdida)
    rsi = rsi.where(perdida != 0, 100.0).where((ganancia != 0) | (perdida != 0), 50.0)
    s['rsi14'] = rsi.where(ganancia.notna())

    macd = cierre.ewm(span=12, adjust=False, min_periods=12).mean() - cierre.ewm(span=26, adjust=False, min_periods=26).mean()
    s['macd_hist'] = (macd - macd.ewm(span=9, adjust=False, min_periods=9).mean()) / cierre
    s['bb_pos'] = ((cierre - s.sma20) / (2 * desv20)).where(desv20 != 0, 0.0).where(s.sma20.notna())

    previo = cierre.shift()
    rango = pd.concat([s.high - s.low, (s.high - previo).abs(), (s.low - previo).abs()], axis=1).max(axis=1)
    s['atr14'] = _wilder(rango, 14) / cierre
    with np.errstate(divide='ignore', invalid='ignore'):
        s['vol_rel'] = np.log(s.volume / s.volume.rolling(20).mean())
    return s


def first_valid_dates(df):
    return {c: df.date[df[c].first_valid_index()] for c in INDICADORES if df[c].first_valid_index() is not None}


def _matriz(df, variables):
    faltantes = [v for v in variables if v not in df.columns]
    if faltantes:
        raise DataError(f'Variables inexistentes: {faltantes}. Disponibles: {list(df.columns)}.')
    return df[variables].to_numpy(dtype=float)


def make_windows(df, variables, lookback):
    """Ejemplos (n, pasos, variables). Cada uno termina en t y su objetivo es el rendimiento de t a t+1."""
    m = _matriz(df, variables)
    if 'y' not in df.columns:
        raise DataError('Falta la columna objetivo y: ejecuta add_target antes de construir ventanas.')
    y = df.y.to_numpy(dtype=float)
    X, Y, inicio, emision, objetivo = [], [], [], [], []
    for t in range(lookback - 1, len(df)):
        ventana = m[t - lookback + 1:t + 1]
        if np.isfinite(ventana).all() and np.isfinite(y[t]):
            X.append(ventana)
            Y.append(y[t])
            inicio.append(df.date.iloc[t - lookback + 1])
            emision.append(df.date.iloc[t])
            objetivo.append(df.date.iloc[t + 1])
    if not X:
        raise DataError(f'Historia insuficiente: {len(df)} filas no alcanzan para una ventana de {lookback} con {variables}.')
    return Ventanas(np.stack(X), np.array(Y), inicio, emision, objetivo)


def latest_window(df, variables, lookback):
    """Ventana de la última vela cerrada, para el pronóstico en vivo. Sin objetivo."""
    m = _matriz(df, variables)
    if len(m) < lookback:
        raise DataError(f'Historia insuficiente: hay {len(m)} velas y la ventana pide {lookback}.')
    x = m[-lookback:]
    if not np.isfinite(x).all():
        malas = [v for i, v in enumerate(variables) if not np.isfinite(x[:, i]).all()]
        raise DataError(f'Valores no finitos en {malas} dentro de la última ventana.')
    return x[None, ...], df.date.iloc[-1]


class Scaler:
    """Estandariza cada variable. Se ajusta solo con TR y después únicamente transforma."""

    def __init__(self, media, desv):
        self.media = np.asarray(media, dtype=float)
        self.desv = np.asarray(desv, dtype=float)

    @classmethod
    def fit(cls, X):
        if X.ndim != 3:
            raise DataError(f'Forma recibida {X.shape}; esperada (ejemplos, pasos, variables).')
        desv = X.std(axis=(0, 1))
        return cls(X.mean(axis=(0, 1)), np.where(desv > 0, desv, 1.0))

    def transform(self, X):
        if X.ndim != 3 or X.shape[2] != len(self.media):
            raise DataError(f'Forma recibida {X.shape}; esperada (ejemplos, pasos, {len(self.media)}).')
        return (X - self.media) / self.desv

    def to_dict(self):
        return {'media': self.media.tolist(), 'desv': self.desv.tolist()}

    @classmethod
    def from_dict(cls, d):
        return cls(d['media'], d['desv'])

In [ ]:
%%writefile bot/model.py
"""Inferencia del modelo aprobado con NumPy. El agente no necesita Keras, GPU ni servicios externos."""
import json
import math
import zipfile
from pathlib import Path

import numpy as np

from bot.data import DataError
from bot.features import Scaler, latest_window

TIPOS = ('cero', 'lineal', 'mlp', 'lstm')
FILTROS = ('ninguno', 'sma20')  # los mismos que evalúa research/lab.py
ACTIVACIONES = {'linear': lambda x: x, 'relu': lambda x: np.maximum(x, 0), 'tanh': np.tanh,
                'sigmoid': lambda x: 1 / (1 + np.exp(-x))}


def _sigmoide(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))


def _lstm(x, w, wr, b):
    """Celda LSTM con las mismas ecuaciones que Keras. Pesos en orden: entrada, olvido, celda, salida."""
    n, pasos, _ = x.shape
    unidades = wr.shape[0]
    h, c = np.zeros((n, unidades)), np.zeros((n, unidades))
    for t in range(pasos):
        z = x[:, t, :] @ w + h @ wr + b
        zi, zf, zc, zo = np.split(z, 4, axis=1)
        i, f, o = _sigmoide(zi), _sigmoide(zf), _sigmoide(zo)
        c = f * c + i * np.tanh(zc)
        h = o * np.tanh(c)
    return h


def adelante(miembro, pesos, indice, x):
    """Propagación hacia adelante de un miembro. x: (ejemplos, pasos, variables)."""
    salida = x
    for j, capa in enumerate(miembro):
        prefijo = f'm{indice}_c{j}_'
        if capa['tipo'] == 'lstm':
            salida = _lstm(salida, pesos[prefijo + 'w'], pesos[prefijo + 'wr'], pesos[prefijo + 'b'])
        else:
            if salida.ndim == 3:
                salida = salida.reshape(len(salida), -1)
            salida = ACTIVACIONES[capa['activacion']](salida @ pesos[prefijo + 'w'] + pesos[prefijo + 'b'])
    return salida.ravel()


def exportar_keras(red, indice):
    """Convierte una red Keras en lista de capas y pesos NumPy. Dropout no actúa al predecir."""
    miembro, pesos = [], {}
    for capa in red.layers:
        nombre = type(capa).__name__
        if nombre in ('Dropout', 'InputLayer'):
            continue
        j = len(miembro)
        prefijo = f'm{indice}_c{j}_'
        if nombre == 'LSTM':
            w, wr, b = capa.get_weights()
            pesos |= {prefijo + 'w': w, prefijo + 'wr': wr, prefijo + 'b': b}
            miembro.append({'tipo': 'lstm'})
        elif nombre == 'Dense':
            w, b = capa.get_weights()
            pesos |= {prefijo + 'w': w, prefijo + 'b': b}
            miembro.append({'tipo': 'dense', 'activacion': capa.get_config()['activation']})
        else:
            raise DataError(f'Capa {nombre} sin exportación a NumPy.')
    return miembro, pesos


def filtro_efectivo(filtro, minimo=''):
    """El filtro de la regla de VA; si no trae ninguno, el mínimo de config/riesgo.json ('' la deja sin filtro)."""
    return (minimo or filtro) if filtro in (None, 'ninguno') else filtro


class Modelo:
    def __init__(self, contrato, pesos):
        self.contrato, self.pesos = contrato, pesos
        self.version = contrato['version']
        self.tipo = contrato['tipo']
        self.lookback = contrato['lookback']
        self.variables = contrato['variables']
        self.umbral = contrato['umbral']
        self.filtro = contrato['filtro']
        self.escalador = Scaler.from_dict(contrato['escalador'])

    def predecir_x(self, x):
        """Pronóstico en la escala original del rendimiento."""
        if self.tipo == 'cero':
            return np.zeros(len(x))
        escalado = self.escalador.transform(x)
        salidas = [adelante(m, self.pesos, i, escalado) for i, m in enumerate(self.contrato['miembros'])]
        return self.contrato['y_media'] + self.contrato['y_desv'] * np.mean(salidas, axis=0)

    def pronosticar(self, tabla):
        if self.tipo == 'cero':
            return 0.0
        x, _ = latest_window(tabla, self.variables, self.lookback)
        return float(self.predecir_x(x)[0])

    def senal(self, pronostico, tabla, filtro_minimo=''):
        """Regla elegida en VA: comprar si el pronóstico supera el umbral y el filtro técnico lo confirma.
        Si la regla no trae filtro, se usa `filtro_minimo` (decisión de Alex, 2026-11-13: ningún activo compra sin filtro)."""
        if not np.isfinite(pronostico):
            raise DataError('Pronóstico no finito: no se puede convertir en señal de trading.')
        ultima = tabla.iloc[-1]
        filtro = filtro_efectivo(self.filtro, filtro_minimo)
        cumple_umbral = bool(pronostico > self.umbral)
        cumple_filtro = True
        if filtro == 'sma20':
            cumple_filtro = bool(ultima.close > ultima.sma20)
        return cumple_umbral and cumple_filtro, dict(umbral=self.umbral, filtro=filtro, cumple_umbral=cumple_umbral,
                                                     cumple_filtro=cumple_filtro)


class ModeloInvalido:
    """Lo que recibe el agente cuyo modelo no pasó la revisión al cargar: queda en pausa y nunca pronostica."""

    def __init__(self, error):
        self.error = error


def _finito(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool) and math.isfinite(x)


def _revisar(contrato, pesos, version):
    """Integridad antes de operar: versión, contrato, escalador y pesos coherentes y finitos. Devuelve el problema o None."""
    if not isinstance(contrato.get('version'), str) or (version is not None and contrato['version'] != version):
        return f'la versión del contrato ({contrato.get("version")}) no coincide con la indicada ({version})'
    variables, lookback = contrato.get('variables'), contrato.get('lookback')
    if not (isinstance(variables, list) and variables and all(isinstance(v, str) for v in variables)):
        return 'la lista de variables está vacía o no es válida'
    if not (isinstance(lookback, int) and not isinstance(lookback, bool) and lookback >= 1):
        return f'la ventana (lookback) no es un entero positivo: {lookback}'
    if contrato.get('filtro') not in FILTROS:
        return f'filtro desconocido: {contrato.get("filtro")}'
    for clave in ('umbral', 'y_media', 'y_desv'):
        if not _finito(contrato.get(clave)):
            return f'{clave} no es un número finito: {contrato.get(clave)}'
    if contrato['y_desv'] <= 0:
        return 'y_desv debe ser positivo'
    escalador = contrato.get('escalador') if isinstance(contrato.get('escalador'), dict) else {}
    try:
        media, desv = np.asarray(escalador.get('media'), dtype=float), np.asarray(escalador.get('desv'), dtype=float)
    except (TypeError, ValueError):
        return 'el escalador no es numérico'
    if media.shape != (len(variables),) or desv.shape != media.shape:
        return f'el escalador no trae un valor por variable ({len(variables)})'
    if not (np.isfinite(media).all() and np.isfinite(desv).all() and (desv > 0).all()):
        return 'el escalador tiene valores no finitos o desviaciones no positivas'
    for k, w in pesos.items():
        if not (np.issubdtype(w.dtype, np.number) and np.isfinite(w).all()):
            return f'el peso {k} tiene valores no finitos o no numéricos'
    if contrato['tipo'] == 'cero':
        return None
    if not contrato.get('miembros'):
        return 'el contrato no declara miembros'
    try:  # prueba de humo: toda la red debe dar un número finito con una ventana neutra
        salida = Modelo(contrato, pesos).predecir_x(np.zeros((1, lookback, len(variables))))
    except (KeyError, ValueError, IndexError, TypeError) as e:
        return f'los pesos no encajan con el contrato ({type(e).__name__}: {e})'
    if salida.shape != (1,) or not np.isfinite(salida).all():
        return 'la red no produce un pronóstico finito'
    return None


def cargar(carpeta, version=None):
    """Carga y revisa el modelo. Con `version` (el nombre de ACTIVO.txt o CARTERA.txt), el contrato debe declarar esa misma versión."""
    carpeta = Path(carpeta)
    ruta = carpeta / 'contrato.json'
    if not ruta.exists():
        raise DataError(f'No existe {ruta}: copia ahí el modelo aprobado o genera uno con research/entrenar_cpu.py.')
    try:
        contrato = json.loads(ruta.read_text(encoding='utf-8'))
    except (OSError, ValueError) as e:
        raise DataError(f'Modelo {carpeta.name} inválido: contrato.json no se puede leer ({type(e).__name__}).') from e
    if not isinstance(contrato, dict):
        raise DataError(f'Modelo {carpeta.name} inválido: contrato.json no es un objeto.')
    if contrato.get('tipo') not in TIPOS:
        raise DataError(f'Tipo de modelo no soportado: {contrato.get("tipo")}. Permitidos: {TIPOS}.')
    pesos = {}
    if contrato['tipo'] != 'cero':
        npz = carpeta / 'modelo.npz'
        if not npz.exists():
            raise DataError(f'Falta {npz}: el contrato pide pesos y no están en la carpeta.')
        try:
            with np.load(npz) as d:
                pesos = {k: d[k] for k in d.files}
        except (OSError, ValueError, EOFError, zipfile.BadZipFile) as e:
            raise DataError(f'Modelo {carpeta.name} inválido: modelo.npz no se puede leer ({type(e).__name__}).') from e
    problema = _revisar(contrato, pesos, version)
    if problema:
        raise DataError(f'Modelo {carpeta.name} inválido: {problema}. No opera hasta reemplazarlo y reiniciar el bot.')
    return Modelo(contrato, pesos)

In [ ]:
%%writefile bot/mexc.py
"""Cliente de datos públicos de MEXC. Solo lectura: aquí no existe ninguna ruta de órdenes."""
import math
import time

import httpx

BASE = 'https://api.mexc.com'
REINTENTOS = 4
VENTANA_MAX_DIAS = 400  # /klines devuelve hasta 500 velas por llamada
DIA_MS = 86_400_000
RUTAS_PERMITIDAS = ('/api/v3/klines', '/api/v3/ticker/price', '/api/v3/ticker/bookTicker', '/api/v3/ticker/24hr', '/api/v3/exchangeInfo', '/api/v3/time', '/api/v3/depth')


class MexcError(RuntimeError):
    pass


class MexcUnavailable(MexcError):
    pass


class MexcRateLimited(MexcError):
    pass


class ParRetirado(MexcError):
    """MEXC ya no lista el par: reintentar no sirve. Reemplazarlo en config/universo.json es decisión de Alex."""


def _niveles(filas):
    """(precio, cantidad) válidos del libro: un nivel con cero, negativo o no numérico se descarta en la frontera."""
    salida = []
    for p, q, *_ in filas:
        precio, cantidad = float(p), float(q)
        if math.isfinite(precio) and math.isfinite(cantidad) and precio > 0 and cantidad > 0:
            salida.append((precio, cantidad))
    return salida


class Mexc:
    def __init__(self, http=None, sleep=time.sleep):
        self.http = http or httpx.Client(base_url=BASE, timeout=15, headers={'User-Agent': 'bot-trading-paper/1.0'})
        self.sleep = sleep

    def _get(self, ruta, params):
        if ruta not in RUTAS_PERMITIDAS:
            raise MexcError(f'{ruta}: este cliente es de solo lectura y no permite esa ruta.')
        url = BASE + ruta
        error = None
        for intento in range(REINTENTOS):
            espera = 2 ** intento
            try:
                r = self.http.get(url, params=params)
            except httpx.HTTPError as e:
                error = MexcUnavailable(f'MEXC no respondió en {ruta} ({type(e).__name__}). Revisa la conexión; se reintentará después.')
            else:
                if r.status_code == 429:
                    error = MexcRateLimited(f'MEXC aplicó límite de solicitudes en {ruta} (HTTP 429).')
                    try:
                        espera = float(r.headers.get('Retry-After', espera))
                    except ValueError:
                        pass
                elif r.status_code >= 500:
                    error = MexcUnavailable(f'MEXC falló en {ruta} con HTTP {r.status_code}.')
                elif r.status_code >= 400:
                    try:
                        codigo = r.json().get('code')
                    except (ValueError, AttributeError):
                        codigo = None
                    if codigo == -1121 and 'symbol' in params:  # «Invalid symbol»: así responde MEXC por un par que ya no lista
                        raise ParRetirado(f'MEXC ya no lista {params["symbol"]}: responde «Invalid symbol» (código -1121).')
                    sufijo = f', código {codigo}' if isinstance(codigo, int) else ''
                    raise MexcError(f'MEXC rechazó {ruta} (HTTP {r.status_code}{sufijo}).')  # su texto no llega al panel ni a la bitácora
                else:
                    try:
                        return r.json()
                    except ValueError as e:
                        raise MexcError(f'Respuesta inesperada de {ruta}: no es JSON.') from e
            if intento < REINTENTOS - 1:
                self.sleep(espera)
        raise error

    def ticker(self, simbolo):
        d = self._get('/api/v3/ticker/price', {'symbol': simbolo})
        try:
            return float(d['price'])
        except (KeyError, TypeError, ValueError) as e:
            raise MexcError(f'Respuesta inesperada del precio de {simbolo}.') from e

    def libro(self, simbolos):
        """Mejor compra y venta de varios activos con una sola consulta, sin reintentos: el siguiente segundo ya es otro intento."""
        ruta = '/api/v3/ticker/bookTicker'
        try:
            r = self.http.get(BASE + ruta)
        except httpx.HTTPError as e:
            raise MexcUnavailable(f'MEXC no respondió en {ruta} ({type(e).__name__}).') from e
        if r.status_code == 429:
            raise MexcRateLimited(f'MEXC aplicó límite de solicitudes en {ruta} (HTTP 429).')
        if r.status_code != 200:
            raise MexcUnavailable(f'MEXC falló en {ruta} con HTTP {r.status_code}.')
        ahora = time.time() * 1000
        quiero = set(simbolos)
        salida = {}
        try:
            for f in r.json():
                if f['symbol'] in quiero and f.get('bidPrice') and f.get('askPrice'):
                    salida[f['symbol']] = dict(bid=float(f['bidPrice']), ask=float(f['askPrice']), ts_ms=ahora)
        except (KeyError, TypeError, ValueError) as e:
            raise MexcError(f'Respuesta inesperada de {ruta}.') from e
        return salida

    def profundidad(self, simbolo, limite=50):
        """Libro visible de un par como (precio, cantidad). Sin reintentos: lo pide el reflejo y el siguiente segundo ya es otro intento."""
        ruta = '/api/v3/depth'
        try:
            r = self.http.get(BASE + ruta, params={'symbol': simbolo, 'limit': limite})
        except httpx.HTTPError as e:
            raise MexcUnavailable(f'MEXC no respondió en {ruta} ({type(e).__name__}).') from e
        if r.status_code == 429:
            raise MexcRateLimited(f'MEXC aplicó límite de solicitudes en {ruta} (HTTP 429).')
        if r.status_code != 200:
            raise MexcUnavailable(f'MEXC falló en {ruta} con HTTP {r.status_code}.')
        try:
            d = r.json()
            return dict(bids=_niveles(d['bids']), asks=_niveles(d['asks']))
        except (KeyError, TypeError, ValueError) as e:
            raise MexcError(f'Respuesta inesperada del libro de {simbolo}.') from e

    def klines(self, simbolo, inicio_ms=None, fin_ms=None, limit=500, intervalo='1d'):
        params = {'symbol': simbolo, 'interval': intervalo, 'limit': limit}
        if inicio_ms is not None:  # MEXC ignora el rango si no llegan ambos extremos
            params |= {'startTime': int(inicio_ms), 'endTime': int(fin_ms)}
        d = self._get('/api/v3/klines', params)
        if not isinstance(d, list):
            raise MexcError(f'Respuesta inesperada de velas de {simbolo}.')
        return d

    def history(self, simbolo, inicio_ms, fin_ms):
        """Historial completo, paginado por ventanas. Conserva tal cual lo que devuelve la fuente."""
        filas = {}
        a = inicio_ms
        while a < fin_ms:
            b = min(a + VENTANA_MAX_DIAS * DIA_MS, fin_ms)
            for f in self.klines(simbolo, a, b):
                filas[f[0]] = f
            a = b + 1
        return [filas[k] for k in sorted(filas)]

    def closed_candles(self, simbolo, limit, now_ms):
        """Solo velas ya cerradas: la vela abierta nunca cuenta como definitiva."""
        fin = int(now_ms)
        inicio = fin - (limit + 2) * DIA_MS
        filas = self.klines(simbolo, inicio, fin, limit=min(limit + 2, 500))
        return [f for f in filas if f[6] <= now_ms][-limit:]

    def filters(self, simbolo):
        d = self._get('/api/v3/exchangeInfo', {'symbol': simbolo})
        try:
            if not d['symbols']:
                raise ParRetirado(f'MEXC ya no lista {simbolo}: exchangeInfo ya no lo incluye.')
            s = d['symbols'][0]
            if str(s.get('status')) != '1':
                raise MexcError(f'{simbolo} no está disponible para operar (estado {s.get("status")}).')
            paso = float(s['baseSizePrecision']) or 10.0 ** -int(s['baseAssetPrecision'])  # MEXC manda '0' en algunos pares
            return {'paso_cantidad': paso, 'minimo_quote': float(s['quoteAmountPrecision']),
                    'maker': float(s.get('makerCommission', 0)), 'taker': float(s.get('takerCommission', 0))}
        except (KeyError, IndexError, TypeError, ValueError) as e:
            raise MexcError(f'Respuesta inesperada de exchangeInfo para {simbolo}.') from e

In [ ]:
%%writefile research/lab.py
"""Laboratorio: preparación, modelos, evaluación y simulación.

El mismo código corre en el notebook de Colab con T4 y en la laptop con CPU. TE solo se abre con abrir_te.
"""
import hashlib
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from bot import data, features, model

UMBRALES = (0.0005, 0.001, 0.002, 0.004, 0.008)  # definidos antes de mirar VA; positivos y distintos del costo
FILTROS = ('ninguno', 'sma20')
CONJUNTOS_VARIABLES = {
    'rezagos': ['ret'],
    'rezagos_indicadores': ['ret', 'dist_sma20', 'vol20'],
}
CONFIGS_MLP = ({'hu': 16, 'lr': 1e-3}, {'hu': 32, 'lr': 1e-3}, {'hu': 32, 'lr': 3e-4})
CONFIGS_LSTM = ({'hu': 8, 'lr': 1e-3}, {'hu': 16, 'lr': 1e-3}, {'hu': 16, 'lr': 3e-4})
SEMILLAS = (11, 22, 33)


# Métricas
def _plano(a, b):
    a, b = np.asarray(a, dtype=float).reshape(-1), np.asarray(b, dtype=float).reshape(-1)
    if len(a) != len(b):
        raise ValueError(f'y y p deben tener la misma longitud: {len(a)} y {len(b)}.')
    if not (np.isfinite(a).all() and np.isfinite(b).all()):
        raise ValueError('y y p deben contener valores finitos.')
    return a, b


def metricas(y, p):
    """Escala original. error = estimado - real. En precisión direccional se excluyen los pares con un cero."""
    y, p = _plano(y, p)
    error = p - y
    validos = (y != 0) & (p != 0)
    largos, cortos = int((p > 0).sum()), int((p < 0).sum())
    denominador = float(((y - y.mean()) ** 2).sum())
    constante = np.std(p) == 0 or np.std(y) == 0
    return dict(MAE=float(np.abs(error).mean()), RMSE=float(np.sqrt((error ** 2).mean())),
                direccional=float((np.sign(y[validos]) == np.sign(p[validos])).mean()) if validos.any() else None,
                R2=1 - float((error ** 2).sum()) / denominador if denominador else None,
                correlacion=None if constante else float(np.corrcoef(y, p)[0, 1]),
                sesgo_modelo=largos / cortos if cortos else None, n=len(y))


def casi_constante(p):
    return bool(np.std(np.asarray(p, dtype=float)) < 1e-5)


def max_drawdown(patrimonio):
    serie = np.asarray(patrimonio, dtype=float)
    picos = np.maximum.accumulate(serie)
    return float(((picos - serie) / picos).max())


# Simulación de la estrategia
def simular(velas, senales, comision, capital=100_000.0):
    """La señal del cierre de t se ejecuta en la apertura de t+1. Solo posición comprada o efectivo."""
    senales = np.asarray(senales, dtype=bool)
    if len(senales) != len(velas):
        raise ValueError(f'{len(senales)} señales para {len(velas)} velas.')
    apertura, cierre = velas.open.to_numpy(dtype=float), velas.close.to_numpy(dtype=float)
    efectivo, cantidad, costos, entrada = float(capital), 0.0, 0.0, 0.0
    resultados, filas = [], []
    for t in range(len(velas)):
        comprar = t > 0 and senales[t - 1]
        ejecucion = None
        if comprar and cantidad == 0:
            nocional = efectivo / (1 + comision)  # incluye el costo al calcular cuánto se puede adquirir
            cantidad = nocional / apertura[t]
            costos += nocional * comision
            entrada, efectivo, ejecucion = efectivo, 0.0, apertura[t]
        elif not comprar and cantidad > 0:
            nocional = cantidad * apertura[t]
            costos += nocional * comision
            efectivo = nocional * (1 - comision)
            resultados.append(efectivo - entrada)
            cantidad, ejecucion = 0.0, apertura[t]
        filas.append(dict(fecha=velas.date.iloc[t], precio_ejecucion=ejecucion, senal=bool(senales[t]), posicion=cantidad,
                          efectivo=efectivo, costos=costos, patrimonio=efectivo + cantidad * cierre[t]))
    if cantidad > 0:  # liquidación final para comparar el cierre del periodo
        nocional = cantidad * cierre[-1]
        costos += nocional * comision
        efectivo = nocional * (1 - comision)
        resultados.append(efectivo - entrada)
        filas[-1] |= dict(precio_ejecucion=cierre[-1], posicion=0.0, efectivo=efectivo, costos=costos, patrimonio=efectivo)
    tabla = pd.DataFrame(filas)
    final = float(tabla.patrimonio.iloc[-1])
    return dict(patrimonio_final=final, rendimiento=final / capital - 1, drawdown=max_drawdown(tabla.patrimonio),
                operaciones=len(resultados), win_rate=float(np.mean([r > 0 for r in resultados])) if resultados else None,
                costos=costos, tabla=tabla)


def comprar_y_mantener(velas, comision, capital=100_000.0):
    """Referencia pasiva desde la misma primera fecha de ejecución y con costos equivalentes."""
    return simular(velas, np.ones(len(velas), dtype=bool), comision, capital)


def elegir_regla(resultados):
    """Mayor rendimiento neto en VA; en empate, menor drawdown."""
    return max(resultados, key=lambda r: (round(r['rendimiento'], 10), -r['drawdown']))


# Partición y preparación
def verificar_particion(fechas_objetivo):
    if not max(fechas_objetivo['TR']) < min(fechas_objetivo['VA']):
        raise ValueError('El último objetivo de TR no es anterior al primero de VA: invalida el experimento.')
    if not max(fechas_objetivo['VA']) < min(fechas_objetivo['TE']):
        raise ValueError('El último objetivo de VA no es anterior al primero de TE: invalida el experimento.')


def folds_expansivos(n_tr, k=3):
    """Ventana expansiva dentro de TR: cada entrenamiento termina antes del bloque que evalúa."""
    bloque = n_tr // (k + 2)
    inicios = [n_tr - (k - i) * bloque for i in range(k)]
    return [(np.arange(0, a), np.arange(a, a + bloque)) for a in inicios]


def tabla_comun(tabla, variables=('ret', 'dist_sma20', 'vol20')):
    """Recorta el inicio para que todas las versiones se evalúen exactamente en las mismas fechas."""
    validas = tabla[list(variables)].notna().all(axis=1)
    return tabla[validas.idxmax():].reset_index(drop=True)


def preparar(tabla, variables, lookback=20):
    """Ventanas, partición por fecha objetivo y escalado ajustado solo con TR."""
    v = features.make_windows(tabla, variables, lookback)
    p = data.split_indices(len(v.y))
    fechas = {k: [v.objetivo[i] for i in idx] for k, idx in p.items()}
    verificar_particion(fechas)
    escalador = features.Scaler.fit(v.X[p['TR']])
    y_media, y_desv = float(v.y[p['TR']].mean()), float(v.y[p['TR']].std())
    salida = dict(variables=variables, lookback=lookback, escalador=escalador, y_media=y_media, y_desv=y_desv, indices=p, ventanas=v)
    for k, idx in p.items():
        salida[f'X_{k}'] = escalador.transform(v.X[idx])
        salida[f'Xcrudo_{k}'] = v.X[idx]
        salida[f'y_{k}'] = v.y[idx]
        salida[f'emision_{k}'] = [v.emision[i] for i in idx]
        salida[f'objetivo_{k}'] = fechas[k]
    return salida


def tabla_particion(d):
    return pd.DataFrame([dict(conjunto=k, ejemplos=len(d[f'y_{k}']), primera_emision=d[f'emision_{k}'][0], ultima_emision=d[f'emision_{k}'][-1],
                              primer_objetivo=d[f'objetivo_{k}'][0], ultimo_objetivo=d[f'objetivo_{k}'][-1]) for k in ('TR', 'VA', 'TE')])


def fila_de_control(d, conjunto='VA', i=0):
    """Una fila completa para explicar por qué el modelo no conoce la respuesta al predecir."""
    indice = d['indices'][conjunto][i]
    v = d['ventanas']
    return dict(entrada_desde=v.inicio[indice], entrada_hasta=v.emision[indice], emision=v.emision[indice], objetivo=v.objetivo[indice],
                rendimiento_objetivo=float(v.y[indice]))


# Modelos clásicos
def ajustar_lineal(X, y, alpha=0.0):
    """Mínimos cuadrados con intercepto. alpha > 0 es Ridge (L2); el intercepto no se penaliza."""
    media_x, media_y = X.mean(axis=0), y.mean()
    Xc = X - media_x
    w = np.linalg.solve(Xc.T @ Xc + (alpha + 1e-9) * np.eye(X.shape[1]), Xc.T @ (y - media_y))
    return w, float(media_y - media_x @ w)


def plano(X):
    return X.reshape(len(X), -1)


def prediccion_lineal(d, conjunto, w, b):
    return plano(d[f'X_{conjunto}']) @ w + b


def ajustar_lineal_modelo(d, alpha=0.0):
    w, b = ajustar_lineal(plano(d['X_TR']), d['y_TR'], alpha)
    return dict(tipo='lineal', w=w, b=b, alpha=alpha)


# Redes (Keras). Se importa aquí para que el agente no dependa de esta librería.
def crear_mlp(entradas, hu, lr, hl=2, dropout=0.0, l2=0.0):
    import keras
    from keras import layers, regularizers
    r = regularizers.L2(l2) if l2 else None
    red = keras.Sequential([layers.Input((entradas,))])
    for _ in range(hl):
        red.add(layers.Dense(hu, activation='relu', kernel_regularizer=r))
        if dropout:
            red.add(layers.Dropout(dropout))
    red.add(layers.Dense(1, activation='linear'))
    red.compile(loss='mse', optimizer=keras.optimizers.Adam(lr))
    return red


def crear_lstm(pasos, variables, hu, lr, dropout=0.0):
    import keras
    from keras import layers
    red = keras.Sequential([layers.Input((pasos, variables)), layers.LSTM(hu)])
    if dropout:
        red.add(layers.Dropout(dropout))
    red.add(layers.Dense(1, activation='linear'))
    red.compile(loss='mse', optimizer=keras.optimizers.Adam(lr))
    return red


def entrenar(red, Xtr, ytr, Xva, yva, semilla, epocas=200, lote=64, paciencia=15):
    """Parada temprana con VA restaurando los mejores pesos. Detiene el proceso si la pérdida no es finita."""
    import keras
    keras.utils.set_random_seed(semilla)
    inicio = time.perf_counter()
    h = red.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=epocas, batch_size=lote, shuffle=True, verbose=0,
                callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=paciencia, restore_best_weights=True),
                           keras.callbacks.TerminateOnNaN()])
    curvas = {k: [float(x) for x in v] for k, v in h.history.items()}
    if not np.isfinite(curvas['loss']).all():
        raise ValueError('Pérdida no finita durante el entrenamiento: revisa escalas, datos e hiperparámetros.')
    return dict(red=red, curvas=curvas, segundos=time.perf_counter() - inicio, mejor_epoca=int(np.argmin(curvas['val_loss'])) + 1,
                parametros=int(red.count_params()))


def construir_red(tipo, d, cfg):
    k = d['X_TR'].shape[2]
    if tipo == 'mlp':
        return crear_mlp(d['X_TR'].shape[1] * k, cfg['hu'], cfg['lr'], dropout=cfg.get('dropout', 0.0), l2=cfg.get('l2', 0.0), hl=cfg.get('hl', 2))
    return crear_lstm(d['X_TR'].shape[1], k, cfg['hu'], cfg['lr'], dropout=cfg.get('dropout', 0.0))


def entradas_red(tipo, X):
    return plano(X) if tipo == 'mlp' else X


def entrenar_red(tipo, d, cfg, semilla, **kw):
    import keras
    keras.utils.set_random_seed(semilla)
    red = construir_red(tipo, d, cfg)
    ytr = (d['y_TR'] - d['y_media']) / d['y_desv']
    yva = (d['y_VA'] - d['y_media']) / d['y_desv']
    return entrenar(red, entradas_red(tipo, d['X_TR']), ytr, entradas_red(tipo, d['X_VA']), yva, semilla, **kw)


def predecir_red(tipo, r, d, conjunto):
    salida = r['red'].predict(entradas_red(tipo, d[f'X_{conjunto}']), verbose=0).ravel()
    return d['y_media'] + d['y_desv'] * salida


# Experimentos
def registrar(experimentos, d, capitulo, variables, modelo, cfg, semilla, tr, va, segundos=0.0, dispositivo='cpu'):
    experimentos.append(dict(capitulo=capitulo, variables=variables, modelo=modelo, configuracion=json.dumps(cfg, ensure_ascii=False), semilla=semilla,
                             dispositivo=dispositivo, segundos=round(segundos, 2), **{f'{k}_TR': v for k, v in metricas(d['y_TR'], tr).items()},
                             **{f'{k}_VA': v for k, v in metricas(d['y_VA'], va).items()}))


def probar_referencias(d, variables, experimentos, dispositivo='cpu'):
    """Referencias: pronóstico cero, regresión lineal y Ridge con los mismos rezagos."""
    registrar(experimentos, d, 7, variables, 'cero', {}, None, np.zeros_like(d['y_TR']), np.zeros_like(d['y_VA']), dispositivo=dispositivo)
    candidatos = []
    for alpha in (0.0, 100.0, 1000.0):
        ajuste = ajustar_lineal_modelo(d, alpha)
        tr, va = prediccion_lineal(d, 'TR', ajuste['w'], ajuste['b']), prediccion_lineal(d, 'VA', ajuste['w'], ajuste['b'])
        registrar(experimentos, d, 7, variables, 'lineal' if not alpha else 'ridge', dict(alpha=alpha), None, tr, va, dispositivo=dispositivo)
        candidatos.append(dict(nombre=f'Lineal alpha={alpha:g}', tipo='lineal', ajuste=ajuste, pred_va=va, variables=variables))
    return candidatos


def probar_red(d, variables, tipo, configs, experimentos, curvas, dispositivo='cpu', semillas=SEMILLAS, **kw):
    """Redes: tres configuraciones elegidas por RMSE de VA; la mejor se repite con tres semillas."""
    mejor = None
    for cfg in configs:
        inicio = time.perf_counter()
        r = entrenar_red(tipo, d, cfg, semillas[0], **kw)
        tr, va = predecir_red(tipo, r, d, 'TR'), predecir_red(tipo, r, d, 'VA')
        registrar(experimentos, d, 8, variables, tipo, {**cfg, 'parametros': r['parametros'], 'mejor_epoca': r['mejor_epoca']}, semillas[0], tr, va, time.perf_counter() - inicio, dispositivo)
        rmse = metricas(d['y_VA'], va)['RMSE']
        if mejor is None or rmse < mejor[0]:
            mejor = (rmse, cfg)
    corridas = []
    for s in semillas:
        inicio = time.perf_counter()
        r = entrenar_red(tipo, d, mejor[1], s, **kw)
        tr, va = predecir_red(tipo, r, d, 'TR'), predecir_red(tipo, r, d, 'VA')
        registrar(experimentos, d, 8, variables, f'{tipo}_mejor', {**mejor[1], 'parametros': r['parametros'], 'mejor_epoca': r['mejor_epoca']}, s, tr, va, time.perf_counter() - inicio, dispositivo)
        corridas.append((r, va))
    rmses = [metricas(d['y_VA'], va)['RMSE'] for _, va in corridas]
    curvas[f'{variables}/{tipo}'] = corridas[0][0]['curvas']
    return dict(nombre=f'{tipo.upper()} ensamble de {len(semillas)} semillas, {mejor[1]["hu"]} neuronas', tipo=tipo, cfg=mejor[1], variables=variables, resultados=[r for r, _ in corridas],
                pred_va=np.mean([va for _, va in corridas], axis=0), rmse_media=float(np.mean(rmses)), rmse_desv=float(np.std(rmses)))


def validacion_expansiva(d, redes=(), **kw):
    """Validación temporal: tres ventanas dentro de TR. Las transformaciones se reajustan solo con cada entrenamiento."""
    v, indices, filas = d['ventanas'], d['indices']['TR'], []
    for n, (a, b) in enumerate(folds_expansivos(len(indices)), 1):
        X_ent, y_ent, X_blo, y_blo = v.X[indices[a]], v.y[indices[a]], v.X[indices[b]], v.y[indices[b]]
        escalador = features.Scaler.fit(X_ent)
        Xe, Xb = escalador.transform(X_ent), escalador.transform(X_blo)
        base = dict(ventana=n, entrenamiento_hasta=v.objetivo[indices[a][-1]], bloque_desde=v.objetivo[indices[b][0]], bloque_hasta=v.objetivo[indices[b][-1]],
                    ejemplos_entrenamiento=len(a), ejemplos_bloque=len(b))
        filas.append({**base, 'modelo': 'cero', **metricas(y_blo, np.zeros_like(y_blo))})
        w, c = ajustar_lineal(plano(Xe), y_ent)
        filas.append({**base, 'modelo': 'lineal', **metricas(y_blo, plano(Xb) @ w + c)})
        for tipo, cfg in redes:
            corte = int(len(Xe) * 0.85)  # la parada temprana usa el tramo final del propio entrenamiento
            y_media, y_desv = float(y_ent.mean()), float(y_ent.std())
            sub = dict(X_TR=Xe[:corte], y_TR=y_ent[:corte], X_VA=Xe[corte:], y_VA=y_ent[corte:], y_media=y_media, y_desv=y_desv)
            r = entrenar_red(tipo, sub, cfg, SEMILLAS[0], **kw)
            pred = y_media + y_desv * r['red'].predict(entradas_red(tipo, Xb), verbose=0).ravel()
            filas.append({**base, 'modelo': tipo, **metricas(y_blo, pred)})
    return filas


# Reglas de señal
def velas_de(tabla, fechas):
    indice = tabla.set_index('date')
    return indice.loc[fechas].reset_index()


def senales(pronostico, velas, umbral, filtro):
    pasa = np.asarray(pronostico) > umbral
    if filtro == 'sma20':
        pasa = pasa & (velas.close.to_numpy() > velas.sma20.to_numpy())
    return pasa


def probar_reglas(pronostico, velas, comision, capital=100_000.0):
    """Hasta cinco umbrales, con y sin filtro técnico. El filtro cambia operaciones, no las predicciones."""
    salida = []
    for filtro in FILTROS:
        for umbral in UMBRALES:
            r = simular(velas, senales(pronostico, velas, umbral, filtro), comision, capital)
            salida.append(dict(umbral=umbral, filtro=filtro, rendimiento=r['rendimiento'], drawdown=r['drawdown'], operaciones=r['operaciones'],
                               win_rate=r['win_rate'], costos=r['costos']))
    return salida


# Sesgo y varianza
def descomposicion(y, predicciones):
    """Sobre reentrenamientos: MSE = (sesgo² + ruido) + varianza. Con datos reales g(x) es desconocida,

    así que sesgo² y ruido solo se reportan juntos. La varianza sí se mide de verdad entre reentrenamientos.
    No confundir con MAE ni con la dispersión de una sola predicción.
    """
    y = np.asarray(y, dtype=float).reshape(-1)
    p = np.asarray(predicciones, dtype=float)
    if p.ndim != 2 or p.shape[1] != len(y):
        raise ValueError(f'Forma recibida {p.shape}; esperada (reentrenamientos, {len(y)}).')
    mse = float(((p - y) ** 2).mean())
    varianza = float(p.var(axis=0).mean())
    return dict(mse=mse, varianza=varianza, sesgo2_mas_ruido=mse - varianza, mse_promedio=float(((p.mean(axis=0) - y) ** 2).mean()), reentrenamientos=len(p))


def bootstrap_bloques(n, rng, largo=20):
    """Remuestreo por bloques para respetar la dependencia temporal al medir varianza."""
    inicios = rng.integers(0, n - largo + 1, size=n // largo + 1)
    return np.concatenate([np.arange(a, a + largo) for a in inicios])[:n]


def diagnostico_clasico(d, ajustar_predecir, reentrenos=12, semilla=7, fraccion=1.0):
    """Reentrena con remuestreos de TR y mide en VA. ajustar_predecir(Xtr, ytr, Xva) -> predicción."""
    rng = np.random.default_rng(semilla)
    Xtr, ytr, Xva = plano(d['X_TR']), d['y_TR'], plano(d['X_VA'])
    inicio = int(len(Xtr) * (1 - fraccion))  # "más datos": menos fracción usa solo el tramo reciente de TR
    Xtr, ytr = Xtr[inicio:], ytr[inicio:]
    pred = []
    for _ in range(reentrenos):
        idx = bootstrap_bloques(len(Xtr), rng)
        pred.append(ajustar_predecir(Xtr[idx], ytr[idx], Xva))
    return descomposicion(d['y_VA'], np.array(pred))


def tacticas_clasicas(d, reentrenos=12):
    """Tácticas de mitigación de la clase, medidas con estos datos. RF y boosting son diagnóstico, no se despliegan."""
    from sklearn.decomposition import PCA
    from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
    from sklearn.linear_model import Lasso
    from sklearn.pipeline import make_pipeline

    def lineal(alpha):
        def f(X, y, Xva):
            w, b = ajustar_lineal(X, y, alpha)
            return Xva @ w + b
        return f

    def sk(fabrica):
        def f(X, y, Xva):
            return fabrica().fit(X, y).predict(Xva)
        return f

    casos = [
        ('Base: regresión lineal', 'Referencia', lineal(0.0), {}),
        ('Agregar más datos (referencia con solo 50% de TR)', 'Agregar más datos', lineal(0.0), {'fraccion': 0.5}),
        ('Ridge L2 alpha=100', 'Regularización', lineal(100.0), {}),
        ('Ridge L2 alpha=1000', 'Regularización', lineal(1000.0), {}),
        ('Lasso L1', 'Regularización', sk(lambda: Lasso(alpha=1e-3, max_iter=5000)), {}),
        ('PCA 5 componentes + lineal', 'Selección de características (PCA)', sk(lambda: make_pipeline(PCA(5), Lasso(alpha=1e-9, max_iter=5000))), {}),
        ('Bagging: Random Forest', 'Bagging', sk(lambda: RandomForestRegressor(n_estimators=40, max_depth=4, n_jobs=-1, random_state=0)), {}),
        ('Boosting: gradiente', 'Boosting', sk(lambda: HistGradientBoostingRegressor(max_depth=2, max_iter=60, learning_rate=0.05, random_state=0)), {}),
    ]
    salida = []
    for nombre, estrategia, funcion, kw in casos:
        r = diagnostico_clasico(d, funcion, reentrenos=reentrenos, **kw)
        salida.append(dict(modelo=nombre, estrategia=estrategia, **r))
    return salida


def tacticas_redes(d, semillas=SEMILLAS, **kw):
    """Complejidad, dropout y L2 en redes: la varianza se mide entre semillas."""
    casos = [
        ('MLP pequeña', 'Referencia de red', 'mlp', dict(hu=8, lr=1e-3)),
        ('MLP grande (más capas y neuronas)', 'Aumentar complejidad', 'mlp', dict(hu=64, lr=1e-3, hl=3)),
        ('MLP grande + Dropout 0.3', 'Regularización', 'mlp', dict(hu=64, lr=1e-3, hl=3, dropout=0.3)),
        ('MLP grande + L2', 'Regularización', 'mlp', dict(hu=64, lr=1e-3, hl=3, l2=1e-3)),
        ('LSTM', 'Red recurrente', 'lstm', dict(hu=16, lr=1e-3)),
    ]
    salida = []
    for nombre, estrategia, tipo, cfg in casos:
        pred = [predecir_red(tipo, entrenar_red(tipo, d, cfg, s, **kw), d, 'VA') for s in semillas]
        salida.append(dict(modelo=nombre, estrategia=estrategia, **descomposicion(d['y_VA'], np.array(pred))))
    return salida


# Entorno del agente futuro (capítulo 10)
def transicion_ejemplo(efectivo, precio_ejecucion, cierre, comision):
    """Una transición calculada a mano: comprar con todo el efectivo y valorar al cierre. La recompensa es el cambio del patrimonio."""
    nocional = efectivo / (1 + comision)
    cantidad = nocional / precio_ejecucion
    costo = nocional * comision
    nuevo = cantidad * cierre
    return dict(cantidad=cantidad, costo=costo, patrimonio_nuevo=nuevo, recompensa=nuevo - efectivo)


def huella(configuracion):
    return hashlib.sha256(json.dumps(configuracion, sort_keys=True, default=str).encode()).hexdigest()


# Sello de apertura de TE
def abrir_te(ruta, huella):
    """TE se evalúa una sola vez con la configuración congelada. Cualquier cambio posterior queda declarado."""
    ruta = Path(ruta)
    sello = json.loads(ruta.read_text(encoding='utf-8')) if ruta.exists() else dict(huella=huella, aperturas=0, contaminado=False)
    sello['aperturas'] += 1
    if huella != sello['huella']:
        sello['contaminado'] = True
    ruta.write_text(json.dumps(sello, indent=2), encoding='utf-8')
    return sello


# Empaquetado del modelo aprobado
def contrato_lineal(d, ajuste):
    """Una regresión lineal se expresa como una capa densa sobre las variables ya escaladas y en y normalizada."""
    w = np.asarray(ajuste['w']).reshape(-1, 1) / d['y_desv']
    b = np.array([(ajuste['b'] - d['y_media']) / d['y_desv']])
    return [[{'tipo': 'dense', 'activacion': 'linear'}]], {'m0_c0_w': w, 'm0_c0_b': b}


def contrato_redes(resultados):
    """Un ensamble promedia varias semillas: bagging de redes para reducir la varianza."""
    miembros, pesos = [], {}
    for i, r in enumerate(resultados):
        m, p = model.exportar_keras(r['red'], i)
        miembros.append(m)
        pesos |= p
    return miembros, pesos


def guardar_modelo(carpeta, version, tipo, d, umbral, filtro, miembros, pesos, extra=None):
    carpeta = Path(carpeta)
    carpeta.mkdir(parents=True, exist_ok=True)
    contrato = dict(version=version, tipo=tipo, lookback=d['lookback'], variables=d['variables'], umbral=umbral, filtro=filtro,
                    escalador=d['escalador'].to_dict(), y_media=d['y_media'], y_desv=d['y_desv'], miembros=miembros, **(extra or {}))
    (carpeta / 'contrato.json').write_text(json.dumps(contrato, indent=2, ensure_ascii=False), encoding='utf-8')
    if pesos:
        np.savez(carpeta / 'modelo.npz', **pesos)
    return model.cargar(carpeta)

In [ ]:
%pip install -q httpx
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from bot import data, features, mexc, model
from research import lab
keras.utils.set_random_seed(SEMILLA_GLOBAL)
np.random.seed(SEMILLA_GLOBAL)
ENTRENAMIENTO = dict(epocas=3, lote=64, paciencia=2) if PRUEBA else dict(epocas=300, lote=64, paciencia=25)
COSTO = 0.001   # costo supuesto: 0.10 % por compra y por venta

## 1. Datos confiables y referencias

**Etapa 1.** Ficha del activo, auditoría, separación TR, VA y TE, y dos modelos de referencia: pronóstico cero y regresión lineal con los mismos rezagos.

In [ ]:
ARCHIVO = Path('BTCUSDT_1d.csv')
if PRUEBA:
    rng = np.random.default_rng(1)
    cierres = 100 * np.cumprod(1 + rng.normal(0.0004, 0.02, 1700))
    dia = 86_400_000
    inicio = 1_500_000_000_000 - 1_500_000_000_000 % dia
    filas = [[inicio + i * dia, str(cierres[i - 1] if i else cierres[0]), str(max(cierres[i], cierres[i - 1] if i else cierres[0]) * 1.01),
              str(min(cierres[i], cierres[i - 1] if i else cierres[0]) * 0.99), str(cierres[i]), '10', inicio + (i + 1) * dia, '100'] for i in range(1700)]
    fuente = 'Datos sintéticos de prueba'
elif ARCHIVO.exists():
    filas = pd.read_csv(ARCHIVO)[data.COLUMNAS].values.tolist()  # copia local identificada y trazable
    fuente = f'Copia local {ARCHIVO}, descargada de MEXC'
else:
    try:
        ahora = int(time.time() * 1000)
        filas = [f for f in mexc.Mexc().history('BTCUSDT', 1_483_228_800_000, ahora) if f[6] <= ahora]
        fuente = mexc.BASE + '/api/v3/klines'
    except mexc.MexcError as e:
        anotar('datos', 'Descarga fallida de MEXC', str(e), 'Subir BTCUSDT_1d.csv de data/raw del proyecto y reejecutar', 'Sin datos')
        raise RuntimeError(f'No se pudo descargar BTCUSDT de MEXC: {e}. Sube el archivo data/raw/BTCUSDT_1d.csv y vuelve a ejecutar.') from e

crudo = data.frame_from_rows(filas)
auditoria = data.audit(crudo)
limpio, limpieza = data.clean(crudo)
crudo.to_csv(SALIDA / 'datos_originales.csv', index=False)
limpio.to_csv(SALIDA / 'datos_procesados.csv', index=False)
for linea in limpieza:
    anotar('preparación', linea, 'Regla de limpieza documentada', 'Conservar el tramo continuo más largo sin rellenar', 'Índice único y orden cronológico verificados')
assert limpio.open_time.is_unique and limpio.open_time.is_monotonic_increasing
if len(limpio) < 1500:
    raise ValueError(f'Solo hay {len(limpio)} observaciones válidas; se necesitan al menos 1,500.')

ficha = dict(activo='BTCUSDT', mercado='Criptomonedas', plataforma='MEXC', fuente=fuente, frecuencia='Diaria', zona_horaria='UTC', calendario='Continuo',
             moneda_base='BTC', moneda_cotizada='USDT', fecha_descarga=time.strftime('%Y-%m-%d'), periodo=[limpio.date.iloc[0], limpio.date.iloc[-1]], observaciones=len(limpio),
             ajustes='Sin ajustes: no hay dividendos ni particiones', volumen='Reportado por la fuente', huella=data.dataset_hash(limpio))
(SALIDA / 'ficha_activo.json').write_text(json.dumps(ficha, indent=2, ensure_ascii=False))
print(json.dumps(ficha, indent=2, ensure_ascii=False))
print('Auditoría del original:', {k: (v if not isinstance(v, list) else (len(v), v[:3])) for k, v in auditoria.items()})
print('Limpieza:', limpieza)

In [ ]:
tabla_total = data.add_target(features.add_indicators(limpio))
tabla = lab.tabla_comun(tabla_total)  # todas las versiones se evalúan en las mismas fechas
fig, ax = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
fechas = pd.to_datetime(tabla.date)
ax[0].plot(fechas, tabla.close); ax[0].plot(fechas, tabla.sma20, '--'); ax[0].set_title('Precio de cierre y SMA20'); ax[0].set_yscale('log')
ax[1].plot(fechas, tabla.ret, linewidth=.6); ax[1].set_title('Rendimiento diario')
ax[2].plot(fechas, tabla.vol20); ax[2].set_title('Volatilidad de 20 días, sin anualizar')
plt.tight_layout(); plt.savefig(SALIDA / 'exploracion.png', dpi=130); plt.show()
extremos = tabla.loc[tabla.ret.abs() > 4 * tabla.ret.std(), ['date', 'ret']]
print('Rendimientos extremos (más de 4 desviaciones):'); print(extremos.to_string(index=False))

**Hipótesis técnica, verificable más adelante:** la distancia del cierre a su SMA de 20 días y la volatilidad de 20 días aportan información para pronosticar el rendimiento del día siguiente. Que una figura se vea sugerente no implica capacidad predictiva: se comprueba en el capítulo 11.

In [ ]:
A = lab.preparar(tabla, lab.CONJUNTOS_VARIABLES['rezagos'])
display(lab.tabla_particion(A))
print('El último objetivo de TR es anterior al primero de VA, y el de VA al primero de TE:', A['objetivo_TR'][-1] < A['objetivo_VA'][0] < A['objetivo_TE'][0])
print('Forma de entrada:', A['X_TR'].shape, '(ejemplos, pasos temporales, variables)')
print('El escalador se ajustó solo con TR. Media y desviación:', A['escalador'].media, A['escalador'].desv)
print('Fila de control:', lab.fila_de_control(A))

**Control de aprendizaje.** La fila anterior muestra que las entradas terminan en la fecha de emisión y que el objetivo es del día siguiente. Al predecir, el modelo solo ve rendimientos ya ocurridos: la respuesta todavía no existe en sus entradas.

In [ ]:
experimentos, curvas = [], {}
DISPOSITIVO = 'cpu (prueba)' if PRUEBA else evidencia['gpu']
candidatos_A = lab.probar_referencias(A, 'rezagos', experimentos)
display(pd.DataFrame(experimentos)[['modelo', 'configuracion', 'MAE_TR', 'RMSE_TR', 'MAE_VA', 'RMSE_VA', 'direccional_VA']])

**Seguimiento.** Para aportar algo, el modelo profundo tendría que reducir el RMSE frente a pronosticar cero en validación. En rendimientos diarios la diferencia suele ser mínima: la mayor parte del error es ruido irreducible.

## 2. MLP y LSTM

**Etapa 2.** Mismo objetivo, variables y particiones. Tres configuraciones por red, variando solo neuronas o tasa de aprendizaje, y la mejor se repite con tres semillas. Parada temprana con VA y restauración de los mejores pesos.

In [ ]:
inicio_redes = time.time()
candidatos_A += [lab.probar_red(A, 'rezagos', 'mlp', lab.CONFIGS_MLP, experimentos, curvas, dispositivo=DISPOSITIVO, **ENTRENAMIENTO),
                 lab.probar_red(A, 'rezagos', 'lstm', lab.CONFIGS_LSTM, experimentos, curvas, dispositivo=DISPOSITIVO, **ENTRENAMIENTO)]
print(f'Entrenamiento en {DISPOSITIVO}: {time.time() - inicio_redes:.0f} s')
if not PRUEBA:
    memoria = tf.config.experimental.get_memory_info('GPU:0')
    print('Memoria de GPU usada, en bytes:', memoria)
    evidencia['memoria_gpu_pico'] = memoria['peak']
    if memoria['peak'] <= 0:
        anotar('entrenamiento', 'La GPU no registró uso', 'El framework entrenó en CPU', 'Detener y revisar el entorno', 'Sin evidencia válida')
        raise RuntimeError('La GPU no se usó durante el entrenamiento: no hay evidencia válida de T4.')
for c in candidatos_A[-2:]:
    print(c['nombre'], f"RMSE VA media={c['rmse_media']:.6f} desviación={c['rmse_desv']:.6f}")
tabla_exp = pd.DataFrame(experimentos)
display(tabla_exp[['modelo', 'configuracion', 'semilla', 'dispositivo', 'segundos', 'RMSE_TR', 'RMSE_VA', 'MAE_VA', 'direccional_VA']])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for eje, (nombre, h) in zip(ax, curvas.items()):
    eje.plot(h['loss'], label='TR'); eje.plot(h['val_loss'], label='VA'); eje.set_title(f'Pérdida por época: {nombre}'); eje.legend()
plt.tight_layout(); plt.savefig(SALIDA / 'curvas_aprendizaje.png', dpi=130); plt.show()
for nombre, h in curvas.items():
    print(nombre, 'mejor época', int(np.argmin(h['val_loss'])) + 1, 'de', len(h['val_loss']))

**Seguimiento.** ¿Una menor pérdida en TR también mejora VA? Compara las curvas: si TR sigue bajando y VA sube, hay sobreajuste; si ambas se quedan altas y planas, el modelo no capta relación útil o casi no la hay en los datos. Escribe aquí lo que observaste en esta corrida.

## 3. Validación temporal y regla de señal

**Etapa 3.** Tres ventanas expansivas dentro de TR: cada entrenamiento termina antes del bloque que evalúa, y las transformaciones se reajustan solo con ese entrenamiento. Después, hasta cinco umbrales candidatos en VA.

El pronóstico directo y recursivo es una ampliación opcional y no se realiza. **Diferenciación fraccionaria:** busca estabilizar una serie sin borrar toda su memoria, con un orden entre 0 y 1. Aquí no es necesaria: los rendimientos ya son estacionarios, como muestra la prueba siguiente; serviría si se usaran precios o volatilidades con memoria larga.

In [ ]:
from statsmodels.tsa.stattools import adfuller
print('Prueba ADF, valor p. Precio:', round(adfuller(tabla.close.values[-1500:])[1], 4), ' Rendimiento:', round(adfuller(tabla.ret.values[-1500:])[1], 6))
redes_A = [(c['tipo'], c['cfg']) for c in candidatos_A if 'cfg' in c]
ventanas_A = lab.validacion_expansiva(A, redes_A, **ENTRENAMIENTO)
tabla_ventanas = pd.DataFrame(ventanas_A)
display(tabla_ventanas.pivot_table(index=['ventana', 'bloque_desde', 'bloque_hasta'], columns='modelo', values='RMSE'))

In [ ]:
velas_va = lab.velas_de(tabla, A['emision_VA'])
reglas_A = []
for c in candidatos_A:
    c['casi_constante'], c['metricas_va'] = lab.casi_constante(c['pred_va']), lab.metricas(A['y_VA'], c['pred_va'])
    c['reglas'] = lab.probar_reglas(c['pred_va'], velas_va, COSTO)
    c['regla'] = lab.elegir_regla(c['reglas'])
    reglas_A += [dict(variables='rezagos', modelo=c['nombre'], **r) for r in c['reglas']]
display(pd.DataFrame([dict(modelo=c['nombre'], RMSE_VA=c['metricas_va']['RMSE'], casi_constante=c['casi_constante'], **{k: c['regla'][k] for k in ('umbral', 'filtro', 'rendimiento', 'drawdown', 'operaciones')}) for c in candidatos_A]))
pasivo_va = lab.comprar_y_mantener(velas_va, COSTO)
print('Comprar y mantener en VA:', round(pasivo_va['rendimiento'], 4), 'caída', round(pasivo_va['drawdown'], 4))

**Seguimiento.** ¿El resultado depende de un único periodo favorable? Revisa si el RMSE se sostiene en las tres ventanas. Un umbral no equivale a una probabilidad de acierto: solo decide cuándo actuar. TE no intervino en ninguna decisión.

## 4. Diseño del agente

**Etapa 4.** Se diseña el entorno; todavía no se entrena un agente DQN o DDQN.

| Elemento | Definición |
| --- | --- |
| Estado | 20 rendimientos recientes, pronóstico, volatilidad, efectivo y posición, todo conocido al cierre de t |
| Acciones | Comprar, mantener una posición abierta, vender una posición existente. Permanecer en efectivo es mantener sin posición |
| Restricciones | Sin apalancamiento, sin posiciones negativas, sin compras mayores al efectivo |
| Recompensa | Cambio del patrimonio después de costos, calculado al cierre del día de ejecución |
| Ejecución | La decisión del cierre de t se ejecuta en la apertura de t+1 |

**Pronóstico, señal y política.** El pronóstico estima un rendimiento. La señal es la instrucción que sale de una regla, como comprar si supera el umbral. La política es lo que aprendería un agente: una función de estados a acciones, entrenada por recompensas.

**DQN y DDQN.** La exploración prueba acciones al azar con probabilidad épsilon para descubrir mejores decisiones. La memoria de experiencias guarda transiciones y las muestrea al azar para romper la correlación temporal. La red objetivo es una copia que se actualiza de vez en cuando para estabilizar el aprendizaje, y DDQN separa quién elige la acción de quién la evalúa para reducir la sobreestimación.

In [ ]:
# Transición calculada: estado, acción, ejecución, recompensa y nuevo estado
t = lab.transicion_ejemplo(efectivo=100_000.0, precio_ejecucion=100.0, cierre=102.0, comision=COSTO)
print('Estado: efectivo 100,000, sin posición. Acción: comprar. Ejecución: apertura de t+1 a 100.')
print('Costo:', round(t['costo'], 2), ' Cantidad:', round(t['cantidad'], 4), ' Patrimonio al cierre a 102:', round(t['patrimonio_nuevo'], 2))
print('Recompensa, cambio del patrimonio después de costos:', round(t['recompensa'], 2))
print('Nuevo estado: efectivo 0, posición', round(t['cantidad'], 4))

**Seguimiento.** ¿El agente podría aprovechar un error del simulador? Sí, si el simulador le dejara operar al cierre que ya conoce, comprar sin costo o gastar más efectivo del disponible. Por eso la ejecución ocurre en la siguiente apertura, los costos se cobran al operar y el saldo se revisa antes de cada compra.

## 5. Indicadores técnicos y evaluación final

**Etapa 5.** El mismo procedimiento con rezagos más dos indicadores: distancia a SMA20 y volatilidad de 20 días. Las dos versiones se comparan en las mismas fechas de VA. La versión final se elige con VA y TE se evalúa una sola vez.

In [ ]:
display(pd.DataFrame([dict(indicador=k, definicion=v, primera_fecha_valida=features.first_valid_dates(tabla_total).get(k)) for k, v in features.INDICADORES.items()]))
B = lab.preparar(tabla, lab.CONJUNTOS_VARIABLES['rezagos_indicadores'])
assert A['objetivo_VA'] == B['objetivo_VA'], 'Las dos versiones deben evaluarse en las mismas fechas.'
candidatos_B = lab.probar_referencias(B, 'rezagos_indicadores', experimentos)
candidatos_B += [lab.probar_red(B, 'rezagos_indicadores', 'mlp', lab.CONFIGS_MLP, experimentos, curvas, dispositivo=DISPOSITIVO, **ENTRENAMIENTO),
                 lab.probar_red(B, 'rezagos_indicadores', 'lstm', lab.CONFIGS_LSTM, experimentos, curvas, dispositivo=DISPOSITIVO, **ENTRENAMIENTO)]
reglas_B = []
for c in candidatos_B:
    c['casi_constante'], c['metricas_va'] = lab.casi_constante(c['pred_va']), lab.metricas(B['y_VA'], c['pred_va'])
    c['reglas'] = lab.probar_reglas(c['pred_va'], velas_va, COSTO)
    c['regla'] = lab.elegir_regla(c['reglas'])
    reglas_B += [dict(variables='rezagos_indicadores', modelo=c['nombre'], **r) for r in c['reglas']]
comparacion = pd.DataFrame([dict(version=v, modelo=c['nombre'], MAE_VA=c['metricas_va']['MAE'], RMSE_VA=c['metricas_va']['RMSE'], direccional_VA=c['metricas_va']['direccional'], casi_constante=c['casi_constante'],
                                 **{k: c['regla'][k] for k in ('umbral', 'filtro', 'rendimiento', 'drawdown', 'operaciones', 'win_rate')})
                            for v, lista in (('rezagos', candidatos_A), ('rezagos_indicadores', candidatos_B)) for c in lista])
display(comparacion)
print(tabla[['ret', 'dist_sma20', 'vol20', 'rsi14', 'macd_hist', 'bb_pos', 'atr14']].corr().round(2).to_string())

In [ ]:
# Comparación C frente a D: la misma estrategia sin filtro y con filtro técnico. El filtro cambia operaciones, no predicciones.
todas_reglas = pd.DataFrame(reglas_A + reglas_B)
filtro = todas_reglas.pivot_table(index=['variables', 'modelo', 'umbral'], columns='filtro', values=['rendimiento', 'drawdown', 'operaciones'])
display(filtro.head(20))

# Selección final con VA, congelada antes de abrir TE
todos = [(A, c) for c in candidatos_A] + [(B, c) for c in candidatos_B]
D, final = max(todos, key=lambda t: (round(t[1]['regla']['rendimiento'], 10), -t[1]['regla']['drawdown']))
referencia_lineal = max(candidatos_A + candidatos_B, key=lambda c: (c['tipo'] == 'lineal', -c['metricas_va']['RMSE']))
configuracion = dict(modelo=final['nombre'], tipo=final['tipo'], variables=D['variables'], regla={k: final['regla'][k] for k in ('umbral', 'filtro')}, datos=ficha['huella'], costo=COSTO,
                     semillas=list(lab.SEMILLAS))
print('Sistema final congelado:', json.dumps(configuracion, indent=2, ensure_ascii=False))
# La exportación a NumPy debe reproducir a Keras con este motor (Colab usa TensorFlow): se comprueba en VA, antes de abrir TE.
previos = lab.contrato_lineal(D, final['ajuste']) if final['tipo'] == 'lineal' else lab.contrato_redes(final['resultados'])
exportado = lab.guardar_modelo(SALIDA / 'verificacion_antes_de_te', 'verificacion', final['tipo'], D, final['regla']['umbral'], final['regla']['filtro'], *previos)
pasos_control = {
    'datos finitos': bool(np.isfinite(D['X_TR']).all() and np.isfinite(D['X_VA']).all() and np.isfinite(D['X_TE']).all()),
    'predicciones VA finitas': bool(np.isfinite(final['pred_va']).all()),
    'fechas alineadas': D['objetivo_VA'][-1] < D['objetivo_TE'][0],
    'transformaciones solo con TR': True,  # lab.preparar ajusta el escalador con TR y solo transforma VA y TE
    'evidencia y registro guardados': (SALIDA / 'ficha_activo.json').exists(),
    'exportación a NumPy reproduce a Keras en VA': bool(np.allclose(exportado.predecir_x(D['ventanas'].X[D['indices']['VA']]), final['pred_va'], atol=1e-4)),
}
print('Controles antes de TE:', pasos_control)
assert all(pasos_control.values()), 'Un control falló: corrige antes de abrir TE.'

In [ ]:
# Única evaluación en TE. Las versiones finales se definieron antes de esta celda.
sello = lab.abrir_te(SALIDA / 'apertura_te.json', lab.huella(configuracion))
print('Sello de TE:', sello)
if sello['contaminado'] or sello['aperturas'] > 1:
    anotar('evaluación', 'TE abierto más de una vez o con otra configuración', 'Reejecución o cambio del sistema', 'Declarar contaminación y reservar un periodo nuevo', 'TE ya no es evaluación intacta')
    print('AVISO: este TE ya no es una evaluación final intacta.')

def predicciones(c, d, conjunto):
    if c['tipo'] == 'lineal':
        return lab.prediccion_lineal(d, conjunto, c['ajuste']['w'], c['ajuste']['b'])
    return np.mean([lab.predecir_red(c['tipo'], r, d, conjunto) for r in c['resultados']], axis=0)

finales = {'Pronóstico cero': (D, None), 'Regresión lineal': (A, next(c for c in candidatos_A if c['nombre'] == 'Lineal alpha=0')),
           'MLP': (A, next(c for c in candidatos_A if c['tipo'] == 'mlp')), 'LSTM': (A, next(c for c in candidatos_A if c['tipo'] == 'lstm')), 'Sistema final': (D, final)}
resultados = []
for nombre, (d, c) in finales.items():
    fila = dict(modelo=nombre)
    for conj in ('TR', 'VA', 'TE'):
        p = np.zeros_like(d[f'y_{conj}']) if c is None else predicciones(c, d, conj)
        m = lab.metricas(d[f'y_{conj}'], p)
        fila |= {f'{k}_{conj}': m[k] for k in ('MAE', 'RMSE', 'direccional')}
    resultados.append(fila)
tabla_final = pd.DataFrame(resultados)
display(tabla_final)
tabla_final.to_csv(SALIDA / 'metricas_finales.csv', index=False)

pred_te = predicciones(final, D, 'TE') if final['tipo'] != 'cero' else np.zeros_like(D['y_TE'])
velas_te = lab.velas_de(tabla, D['emision_TE'])
senal_te = lab.senales(pred_te, velas_te, final['regla']['umbral'], final['regla']['filtro'])
pd.DataFrame(dict(emision=D['emision_TE'], objetivo=D['objetivo_TE'], real=D['y_TE'], estimado=pred_te, error=pred_te - D['y_TE'], senal=np.where(senal_te, 'comprar', 'efectivo'))).to_csv(SALIDA / 'predicciones_TE.csv', index=False)

sim = lab.simular(velas_te, senal_te, COSTO)
pasivo = lab.comprar_y_mantener(velas_te, COSTO)
sim['tabla'].to_csv(SALIDA / 'simulacion.csv', index=False)
resumen_sim = pd.DataFrame([
    dict(estrategia='Sistema final', **{k: sim[k] for k in ('patrimonio_final', 'rendimiento', 'drawdown', 'operaciones', 'win_rate', 'costos')}),
    dict(estrategia='Comprar y mantener', **{k: pasivo[k] for k in ('patrimonio_final', 'rendimiento', 'drawdown', 'operaciones', 'win_rate', 'costos')}),
    dict(estrategia='Efectivo', patrimonio_final=100_000.0, rendimiento=0.0, drawdown=0.0, operaciones=0, win_rate=None, costos=0.0)])
display(resumen_sim)
resumen_sim.to_csv(SALIDA / 'resumen_simulacion.csv', index=False)
plt.figure(figsize=(11, 3.6)); plt.plot(pd.to_datetime(velas_te.date), sim['tabla'].patrimonio, label='Sistema final'); plt.plot(pd.to_datetime(velas_te.date), pasivo['tabla'].patrimonio, '--', label='Comprar y mantener')
plt.title('Patrimonio en TE con costo de 0.10 % por operación'); plt.legend(); plt.savefig(SALIDA / 'patrimonio_te.png', dpi=130); plt.show()

In [ ]:
# Pronóstico del siguiente periodo no observado. El valor real queda vacío hasta observarse.
tabla_viva = features.add_indicators(limpio)
x, corte = features.latest_window(tabla_viva, D['variables'], D['lookback'])
xz = D['escalador'].transform(x)
if final['tipo'] == 'lineal':
    futuro = float((lab.plano(xz) @ final['ajuste']['w'] + final['ajuste']['b'])[0])
else:
    futuro = float(np.mean([D['y_media'] + D['y_desv'] * r['red'].predict(lab.entradas_red(final['tipo'], xz), verbose=0).ravel()[0] for r in final['resultados']]))
senal_futura = lab.senales(np.array([futuro]), tabla_viva.iloc[[-1]], final['regla']['umbral'], final['regla']['filtro'])[0]
pd.DataFrame([dict(fecha_corte=corte, proximo_periodo=str((pd.Timestamp(corte) + pd.Timedelta(days=1)).date()), rendimiento_estimado=futuro, senal='comprar' if senal_futura else 'efectivo', valor_real='')]).to_csv(SALIDA / 'prediccion_futura.csv', index=False)
print('Corte:', corte, ' Pronóstico:', round(futuro, 5), ' Señal:', 'comprar' if senal_futura else 'efectivo')

**Seguimiento.** ¿El indicador aporta información o solo complejidad? Compara las dos versiones en la tabla anterior y responde con los datos de esta corrida. Revisa también las correlaciones: variables casi iguales añaden redundancia, no información. Rendimientos históricos, incluso en TE, no garantizan resultados futuros.

## Manejo de errores: cuatro fallos provocados

Se usan copias pequeñas: los datos originales no se tocan. Cada fallo debe detenerse con un mensaje que explique qué revisar.

In [ ]:
def provocar(nombre, funcion):
    try:
        funcion()
        print(f'{nombre}: NO se detuvo.'); raise AssertionError(nombre)
    except (data.DataError, ValueError) as e:
        print(f'{nombre}: detenido. Mensaje: {e}')
        anotar('prueba controlada', nombre, 'Fallo provocado en una copia', 'El flujo se detuvo con mensaje claro', 'Datos originales sin cambios')

copia = limpio.head(60).copy()
provocar('Conjunto vacío', lambda: data.frame_from_rows([]))
duplicada = pd.concat([copia, copia.tail(1)])
informe = data.audit(duplicada)
print('Fecha duplicada detectada:', informe['duplicadas'], 'fila; la limpieza la corrige:', data.clean(duplicada)[1])
anotar('prueba controlada', 'Fecha duplicada', 'Fila repetida en una copia', 'Eliminada según regla documentada', 'Índice único verificado')
invalida = copia.copy(); invalida.loc[invalida.index[-5], 'close'] = -1
print('Valor no válido detectado:', data.audit(invalida)['invalidos'])
anotar('prueba controlada', 'Valor no válido', 'Precio negativo en una copia', 'Identificada columna y fecha; fila retirada sin usar información futura', 'Sin precios no positivos')
provocar('Dimensiones incorrectas', lambda: A['escalador'].transform(np.ones((4, 20))))
provocar('Ventanas insuficientes', lambda: features.make_windows(data.add_target(features.add_indicators(copia.head(10))), ['ret'], 20))
provocar('Predicción no finita', lambda: lab.metricas(np.array([np.nan]), np.array([0.0])))
print('Datos originales intactos:', data.dataset_hash(limpio) == ficha['huella'])

## Guardar resultados y modelo

In [ ]:
carpeta_modelo = SALIDA / f"t4-{limpio.date.iloc[-1]}" if not PRUEBA else SALIDA / 'prueba-sin-t4'
version = carpeta_modelo.name
if final['tipo'] == 'lineal':
    miembros, pesos = lab.contrato_lineal(D, final['ajuste'])
else:
    miembros, pesos = lab.contrato_redes(final['resultados'])
    for i, r in enumerate(final['resultados']):
        carpeta_modelo.mkdir(parents=True, exist_ok=True)
        r['red'].save(carpeta_modelo / f'red_{i}.keras')
seleccion = dict(periodo_va=[D['emision_VA'][0], D['emision_VA'][-1]], modelo=final['nombre'], variables=configuracion['variables'], regla=final['regla'], metricas_va=final['metricas_va'],
                 comprar_y_mantener_va=dict(rendimiento=pasivo_va['rendimiento'], drawdown=pasivo_va['drawdown']), rmse_cero_va=lab.metricas(D['y_VA'], np.zeros_like(D['y_VA']))['RMSE'],
                 particion=lab.tabla_particion(D).to_dict('records'))
m = lab.guardar_modelo(carpeta_modelo, version, final['tipo'], D, final['regla']['umbral'], final['regla']['filtro'], miembros, pesos,
                       extra=dict(valido_para_entrega=not PRUEBA, dispositivo=DISPOSITIVO, huella_datos=ficha['huella'], nombre=final['nombre'], conjunto_variables=[k for k, v in lab.CONJUNTOS_VARIABLES.items() if v == D['variables']][0], seleccion=seleccion))
assert np.allclose(m.predecir_x(D['ventanas'].X[D['indices']['VA']]), final['pred_va'], atol=1e-4), 'El modelo exportado no reproduce las predicciones.'
print('Modelo exportado y verificado: NumPy reproduce las predicciones de Keras.')

experimentos_df = pd.DataFrame(experimentos)
experimentos_df.insert(0, 'id', range(1, len(experimentos_df) + 1))
experimentos_df.to_csv(SALIDA / 'experimentos.csv', index=False)
experimentos_df.drop(columns='id').to_csv(carpeta_modelo / 'experimentos.csv', index=False)
todas_reglas.to_csv(carpeta_modelo / 'reglas_va.csv', index=False)
tabla_ventanas.to_csv(SALIDA / 'ventanas_expansivas.csv', index=False)
tabla_ventanas.to_csv(carpeta_modelo / 'ventanas_expansivas.csv', index=False)
(carpeta_modelo / 'curvas.json').write_text(json.dumps(curvas))
print('Diagnóstico de sesgo y varianza...')
diagnostico = lab.tacticas_clasicas(D, reentrenos=3 if PRUEBA else 12) + lab.tacticas_redes(D, **ENTRENAMIENTO)
(carpeta_modelo / 'diagnostico.json').write_text(json.dumps(diagnostico, indent=2, ensure_ascii=False))
display(pd.DataFrame(diagnostico))
pd.DataFrame(bitacora).to_csv(SALIDA / 'bitacora_errores.csv', index=False)
evidencia |= dict(fin=time.strftime('%Y-%m-%d %H:%M:%S'), segundos_totales=round(time.time() - inicio_notebook), dispositivo_entrenamiento=DISPOSITIVO)
(SALIDA / 'evidencia_t4.json').write_text(json.dumps(evidencia, indent=2, ensure_ascii=False))
(carpeta_modelo / 'evidencia_t4.json').write_text(json.dumps(evidencia, indent=2, ensure_ascii=False))

In [ ]:
with zipfile.ZipFile('resultados_t4.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for ruta in SALIDA.rglob('*'):
        z.write(ruta, ruta.relative_to(SALIDA))
print('Listo:', Path('resultados_t4.zip').stat().st_size // 1024, 'KB')
print('Carpeta del modelo para copiar a models/ del proyecto:', carpeta_modelo.name)
if DRIVE:
    import shutil
    shutil.copy('resultados_t4.zip', DRIVE / f'resultados_t4_{time.strftime("%Y%m%d_%H%M")}.zip')
    print('Copia guardada en Drive:', DRIVE)
if not PRUEBA:
    from google.colab import files
    files.download('resultados_t4.zip')

## Lectura de resultados

Cada etapa responde qué se hizo, por qué, qué evidencia salió y qué cambiaría. La conclusión sobre la hipótesis técnica sale de la etapa 5. Las cifras deben salir de esta corrida, no copiarse de otra, y el notebook debe correr completo desde una sesión limpia.